# G v2 한 단계씩 따라 하기

이 노트북은 **깜빡임 없는 영상 하나**에 G(생성기)를 적용해서, 깜빡임이 들어간 학습용 영상이 어떻게 만들어지는지 한 단계씩 보여 줍니다.

```
① 영상 고르기 → ② G가 쓰는 크기로 바꾸기 → ③ 깜빡임 넣을 영역(ROI) 정하기
→ ④ G 불러오기 → ⑤ G 적용 → ⑥ 판정(표준 문턱을 넘었나) → ⑦ 영상으로 보기
```

**실행 방법:** 위에서부터 셀을 하나씩 실행하세요(셀 선택 후 `Shift + Enter`). 커널은 **LUMI (_work/.venv)**를 고르세요.

> ⚠ ⑦번 셀의 영상에는 광과민성 발작을 일으킬 수 있는 깜빡임이 있습니다. 광과민성이 있거나 의심되면 ⑦번은 실행하지 마세요. ①~⑥은 깜빡이는 영상을 재생하지 않습니다.

## 0. 준비
필요한 도구를 불러오고, 저장소·데이터·학습된 G의 위치를 찾습니다. 그래픽카드(GPU)가 있으면 씁니다.

In [2]:
import json
from pathlib import Path

import cv2
import numpy as np
import torch
import matplotlib.pyplot as plt
plt.rcParams["font.family"] = "Malgun Gothic"     # 그래프 안 한글이 네모로 깨지지 않게 (Windows 기본 한글 글꼴)
plt.rcParams["axes.unicode_minus"] = False

# 저장소 폴더 찾기 (이 노트북이 저장소 안 어디에 있든 동작)
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "lumi").exists())
import sys; sys.path.insert(0, str(REPO / "src"))
from lumi.gan_v2 import GeneratorV2, TARGET_FAIL, TARGET_NEAR
from lumi.judge import judge_luminance_flash, srgb_to_luminance_np

WORK = REPO.parent / "_work"
CHECKPOINT = WORK / "checkpoints" / "g_v2_e2b.pt"      # 학습된 G (실험 E2b)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("저장소:", REPO)
print("G 파일:", CHECKPOINT, "있음" if CHECKPOINT.exists() else "없음!")
print("장치:", DEVICE)

ModuleNotFoundError: No module named 'cv2'

## ① 영상 고르기
`VIDEO`에 **깜빡임이 없는** 영상 파일 경로를 넣으세요. 비워 두면(`None`) 학습에 쓰지 않은 실제 안전 GIF 중 하나(`BG_INDEX`번)를 씁니다.

In [ ]:
VIDEO = None          # 예: r"D:\내영상.mp4"
BG_INDEX = 0          # VIDEO가 None일 때 쓸 검증용 배경 번호 (0~203)

## ② G가 쓰는 크기로 바꾸기
G는 **가로 128 × 세로 96, 초당 24장, 32장(약 1.3초)** 짜리 영상만 받습니다. 그래서 영상의 가운데를 4:3으로 잘라 줄이고, 시간도 24장/초로 맞춥니다. 영상이 짧으면 반복해서 채웁니다.

아래 그림은 바뀐 영상에서 4장마다 한 장씩 뽑은 **정지 사진**입니다(깜빡임 없음).

In [ ]:
FPS, FRAMES, W, H = 24.0, 32, 128, 96

def load_clip(path, fps=FPS, frames=FRAMES, w=W, h=H):
    cap = cv2.VideoCapture(str(path)); fps_in = cap.get(cv2.CAP_PROP_FPS) or 10.0; src = []
    while len(src) < 2000:
        ok, f = cap.read()
        if not ok: break
        src.append(f)
    cap.release()
    h0, w0 = src[0].shape[:2]
    cw, ch = (h0 * 4 // 3, h0) if w0 * 3 > h0 * 4 else (w0, w0 * 3 // 4)   # 가운데 4:3
    x0, y0 = (w0 - cw) // 2, (h0 - ch) // 2
    dur = len(src) / fps_in
    out = [cv2.cvtColor(cv2.resize(src[min(len(src) - 1, int(((i / fps) % dur) * fps_in))][y0:y0 + ch, x0:x0 + cw],
                                   (w, h), interpolation=cv2.INTER_AREA), cv2.COLOR_BGR2RGB) for i in range(frames)]
    return np.stack(out)

if VIDEO:
    clean = load_clip(VIDEO); name = Path(VIDEO).name
else:
    clean = np.load(WORK / "datasets" / "real_bg_v1" / "val.npy", mmap_mode="r")[BG_INDEX].copy()
    name = f"검증용 배경 {BG_INDEX}번"

print(name, "→ 모양", clean.shape, "(프레임, 세로, 가로, 색)")
fig, ax = plt.subplots(1, 8, figsize=(16, 2))
for k, a in enumerate(ax):
    a.imshow(clean[k * 4]); a.set_title(f"{k * 4 + 1}번째"); a.axis("off")
plt.show()

## ③ 깜빡임 넣을 영역(ROI) 정하기
G는 **ROI 안에서만** 영상을 바꿉니다. ROI 밖은 원본 그대로 둡니다(국소 보정 연구용 데이터라서).

`ROI = (왼쪽, 위, 오른쪽, 아래)`를 화면 비율(0~1)로 정하세요. 판정 규칙이 "화면의 25% 이상이 깜빡여야 위험"이라서, ROI가 너무 작으면 문턱을 넘길 수 없습니다(30% 이상 권장).

In [ ]:
ROI = (0.17, 0.17, 0.83, 0.83)       # 가운데 약 44%

mask = np.zeros((H, W), np.uint8)
mask[int(ROI[1] * H):int(ROI[3] * H), int(ROI[0] * W):int(ROI[2] * W)] = 1
print(f"ROI 면적: 화면의 {mask.mean():.0%}")

show = clean[0].copy(); show[mask == 0] = (show[mask == 0] * 0.35).astype(np.uint8)   # ROI 밖을 어둡게
plt.figure(figsize=(4, 3)); plt.imshow(show); plt.title("밝은 부분 = ROI"); plt.axis("off"); plt.show()

## ④ G 불러오기
학습해 둔 G(실험 E2b)를 불러옵니다. 이 G는

- **목표**만 받습니다: `문턱 넘기`(표준상 위험) 또는 `문턱 근처`(깜빡이지만 표준상 통과)
- 어떻게 깜빡일지(주파수·세기)는 **z**라는 무작위 숫자에 따라 G가 스스로 정합니다
- 원본에 **밝기 배율과 색 보정**을 곱하는 방식이라 배경의 질감이 남습니다

In [ ]:
ck = torch.load(CHECKPOINT, map_location=DEVICE, weights_only=False)
cfg = ck["config"]
G = GeneratorV2(cfg["width"], cfg["zdim"], cfg["frames"], cfg["fps"], cfg.get("output", "residual")).to(DEVICE).eval()
G.load_state_dict(ck["generator"])
print("G 준비 완료 · 출력 방식:", cfg.get("output"), "· 파라미터", round(sum(p.numel() for p in G.parameters()) / 1e6, 2), "M개")

## ⑤ G 적용
`SEED`를 바꾸면 z가 바뀌어서 깜빡이는 방식이 달라집니다. 두 목표(`문턱 근처`, `문턱 넘기`)로 각각 만듭니다.

In [ ]:
SEED = 0

torch.manual_seed(SEED)
z = torch.randn(1, cfg["zdim"], device=DEVICE)
x = torch.from_numpy(clean).to(DEVICE).permute(3, 0, 1, 2).float()[None] / 255
m = torch.from_numpy(mask).to(DEVICE).float()[None, None, None].expand(1, 1, FRAMES, H, W)

outs = {}
for key, target in (("문턱 근처", TARGET_NEAR), ("문턱 넘기", TARGET_FAIL)):
    with torch.no_grad():
        o = G(x, m, torch.tensor([target], device=DEVICE), z)
    clip = np.rint(o[0].permute(1, 2, 3, 0).float().cpu().numpy() * 255).astype(np.uint8)
    outs[key] = np.where(mask[None, :, :, None] > 0, clip, clean)     # ROI 밖은 원본 그대로
print("만든 영상:", {k: v.shape for k, v in outs.items()})

## ⑥ 판정: 표준 문턱을 넘었나
판정 규칙(WCAG·ITU의 일반 깜빡임, `src/lumi/judge.py`):

1. 픽셀마다 밝기(상대 휘도)가 앞 프레임보다 **0.1 이상** 바뀌고, 어두운 쪽이 0.8 미만이면 "바뀐 픽셀"
2. 같은 방향으로 바뀐 픽셀이 **화면의 25% 이상**이면 그 순간은 "전이" 한 번
3. **1초 안에 전이가 6번을 넘으면(깜빡임 3회 초과) 실패** = 표준상 위험

아래 그래프는 ROI 안의 평균 밝기가 시간에 따라 어떻게 바뀌는지와, 판정기가 "전이"로 센 순간(세로선)입니다.

In [ ]:
clips = {"원본": clean, **outs}
fig, ax = plt.subplots(1, 3, figsize=(16, 3.2), sharey=True)
for a, (key, c) in zip(ax, clips.items()):
    j = judge_luminance_flash(c, FPS)
    y = srgb_to_luminance_np(c)[:, mask > 0].mean(axis=1)
    a.plot(np.arange(FRAMES) / FPS, y, marker="o", ms=3)
    for f in j.transition_frames:
        a.axvline(f / FPS, color="orange", alpha=0.5)
    verdict = "실패(위험)" if j.fail else "통과"
    a.set_title(f"{key}: {verdict} · 1초 최대 전이 {j.max_transitions_1s}회")
    a.set_xlabel("시간(초)")
ax[0].set_ylabel("ROI 평균 밝기")
plt.tight_layout(); plt.show()

## ⑦ 영상으로 보기
> ⚠ **깜빡임이 있습니다.** 광과민성이 있거나 의심되면 실행하지 마세요. 작은 창, 밝은 방, 한쪽 눈 가리기를 권장합니다.

자동으로 재생되지 않습니다. 플레이어 아래 버튼으로 **한 프레임씩(◀▶ 단추)** 넘겨 보거나 재생하세요. 왼쪽부터 원본 | 문턱 근처 | 문턱 넘기입니다.

In [ ]:
from matplotlib import animation
from IPython.display import HTML

plt.rcParams["animation.embed_limit"] = 50
fig, ax = plt.subplots(1, 3, figsize=(12, 3.4))
ims = []
for a, (key, c) in zip(ax, clips.items()):
    ims.append(a.imshow(c[0])); a.set_title(key); a.axis("off")
def step(i):
    for im, c in zip(ims, clips.values()):
        im.set_data(c[i])
    return ims
anim = animation.FuncAnimation(fig, step, frames=FRAMES, interval=1000 / FPS)
plt.close(fig)
HTML(anim.to_jshtml(default_mode="loop"))

## 바꿔 보기
- 다른 깜빡임 방식: ⑤의 `SEED`를 바꾸고 ⑤ → ⑥ → ⑦ 다시 실행
- 다른 위치·크기: ③의 `ROI`를 바꾸고 ③부터 다시 실행
- 다른 영상: ①의 `VIDEO` 또는 `BG_INDEX`를 바꾸고 ①부터 다시 실행

판정은 **표준 문턱 판정**이지 실제로 발작을 일으키는지에 대한 판정이 아닙니다.